In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

tmin = 0
tmax=0.8
fmin=0.5
fmax = 16
sfreq = 32

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
    #BI2012()
    #bi2013a(),
    #bi2014a(),
    #bi2014b(),
    #bi2015a(),
    #bi2015b(),
    BNCI2014008(),
    BNCI2014009(),
    #BNCI2015003(),
    #Cattan2019_VR()
    #EPFLP300(),
    #Huebner2017(),
    #Huebner2018(),
    #Lee2019_ERP(),
    #Sosulski2019()

]
evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    suffix="hoda",
    overwrite=False,
    random_state=42,
    n_jobs=-1,
    error_score=0
    #data_size=dict(
    #    policy='per_class',
    #    value=[10]
    #),
    #n_perms=[2],
)

In [3]:
import tensorly as tl
tl.set_backend('cupy', local_threadsafe=False)

In [4]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from toeplitzlda.classification import ToeplitzLDA


def reshape(X, y=None):
    return tl.to_numpy(X.reshape((X.shape[0],-1)))
    
class ToeplitzLDAWrapper(BaseEstimator, ClassifierMixin):

    def fit(self, X, y=None):
        self.classes_ = np.unique(y)
        n_epochs, n_channels, n_samples = X.shape
        self.tlda_ = ToeplitzLDA(n_channels=n_channels,
                                 data_is_channel_prime=False)
        return self.tlda_.fit(reshape(X), y)

    def decision_function(self, X):
        n_epochs, n_channels, n_samples = X.shape
        return self.tlda_.decision_function(reshape(X))

    def predict(self, X):
        return self.tlda_.predict(reshape(X))

    def predict_proba(self, X):
        return self.tlda_.predict_proba(reshape(X))




In [5]:
from sklearn.pipeline import make_pipeline
from mne.decoding import Scaler
from hoda.hoda import HODA, BTTDA
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from pyriemann.estimation import XdawnCovariances
from pyriemann.tangentspace import TangentSpace

pipelines = dict()

pipelines['HODA_rt'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    HODA(
        max_iter=128,
        rank=None,
        tol=1e-8,
        init ='svd',
        shrinkage=('lw','lw'),
        toeplitz=None,
        obj='rt',
        solver='lanczos',        
        verbose=False,
        taper=False,
        lasso=False,
        prune=True,
        prune_pvalue=0.05,
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)

pipelines['BTTDA_rt_2'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=2,
        hoda_params=dict(
            max_iter=128,
            rank=None,
            tol=1e-8,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            prune_pvalue=0.05,
            keep_train_info=False
        ),
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)

pipelines['BTTDA_rt_4'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=4,
        hoda_params=dict(
            max_iter=128,
            rank=None,
            tol=1e-8,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            prune_pvalue=0.05,
            keep_train_info=False
        ),
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)

pipelines['BTTDA_rt_6'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=6,
        hoda_params=dict(
            max_iter=128,
            rank=None,
            tol=1e-8,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            prune_pvalue=0.05,
            keep_train_info=False
        ),
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)
pipelines['BTTDA_rt_8'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=8,
        hoda_params=dict(
            max_iter=128,
            rank=None,
            tol=1e-8,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            prune_pvalue=0.05,
            keep_train_info=False
        ),
            deflate_transform=False,

        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)

pipelines['BTTDA_rt_8_defl'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=8,
        hoda_params=dict(
            max_iter=128,
            rank=None,
            tol=1e-8,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            prune_pvalue=0.05,
            keep_train_info=False
        ),
            deflate_transform=True,

        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)
pipelines['BTTDA_rt_16'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=16,
        hoda_params=dict(
            max_iter=128,
            rank=None,
            tol=1e-8,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            prune_pvalue=0.05,
            keep_train_info=False
        ),
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)
pipelines['BTTDA_rt_16_var-expl'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=16,
        hoda_params=dict(
            max_iter=128,
            rank=None,
            tol=1e-8,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            prune_pvalue=0.05,
            keep_train_info=False
        ),
        var_thresh=.75,
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)
pipelines['HODA_tr'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    HODA(
        max_iter=128,
        rank=None,
        tol=1e-8,
        init ='svd',
        shrinkage=('lw','lw'),
        toeplitz=None,
        obj='tr',
        solver='lanczos',        
        verbose=False,
        taper=False,
        lasso=False,
        prune=True,
        prune_pvalue=0.05,
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    StandardScaler(),
    LogisticRegressionCV(Cs=10, penalty='l1', solver='liblinear', class_weight='balanced', scoring='roc_auc', n_jobs=-1),
)

pipelines['tLDA'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    ToeplitzLDAWrapper()
)

pipelines['sLDA'] = make_pipeline(
        Scaler(scalings='mean', with_mean=False),
        FunctionTransformer(reshape),
        LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)


pipelines['XDAWN+RG'] = make_pipeline(
    XdawnCovariances(n_components=3),
    TangentSpace(metric="riemann"),
    LogisticRegression(),
)


In [ ]:
#import warnings
#warnings.filterwarnings("ignore")

results = evaluation.process(pipelines)

008-2014-WithinSession:   0%|                                                                   | 0/8 [00:00<?, ?it/s]/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to conv

Fitted Tucker model of rank [5, 5] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [4, 4] ...
Fitted Tucker model of rank [2, 2] ...


/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fitted Tucker model of rank [5, 5] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...


/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fitted Tucker model of rank [5, 5] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [4, 4] ...
Fitted Tucker model of rank [2, 2] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [4, 4] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...


/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/s

Fitted Tucker model of rank [5, 5] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...
Fitted Tucker model of rank [1, 1] ...


/home/arne/.virtualenvs/hoda/lib64/python3.9/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [ ]:
results

In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
from moabb.analysis.plotting import meta_analysis_plot, paired_plot
_ = meta_analysis_plot(stats, 'BTTDA_rt_16', 'BTTDA_rt_8')
_  = paired_plot(results, 'BTTDA_rt_16', 'BTTDA_rt_8')

In [ ]:
import seaborn as sns
from moabb.analysis.meta_analysis import (
    collapse_session_scores,
    combine_effects,
    combine_pvalues,
)
import matplotlib.pyplot as plt

from matplotlib.gridspec import GridSpec



plt.style.use('default')
def paired_plot(data, alg1, alg2, ax, **kwargs):
    """Generate a figure with a paired plot.

    Parameters
    ----------
    data: DataFrame
        dataframe obtained from evaluation
    alg1: str
        Name of a member of column data.pipeline
    alg2: str
        Name of a member of column data.pipeline

    Returns
    -------
    fig: Figure
        Pyplot handle
    """
    cmap = ['#ef5513', '#f9c134']
    data = collapse_session_scores(data)
    data = data[data.pipeline.isin([alg1, alg2])]
    data = data.pivot_table(
        values="score", columns="pipeline", index=["subject", "dataset"]
    )
    data = data.reset_index()
    sns.scatterplot(data=data, x=alg1, y=alg2, ax=ax, hue='dataset', palette=cmap, **kwargs)
    ax.plot([0, 1], [0, 1], ls="--", c="grey")
    ax.set_xlim([0.75, 1])
    ax.set_ylim([0.75, 1])
    return fig

fig = plt.figure()

gs = GridSpec(2,3, figure=fig, width_ratios=[2.5,1,1],)
ax1 = fig.add_subplot(gs[:,0])
ax2 = fig.add_subplot(gs[0,1])
ax3 = fig.add_subplot(gs[0,2])
ax4 = fig.add_subplot(gs[1,1])

_=paired_plot(results, "HODA_rt", "BTTDA_rt_8", ax1, legend=True)
ax1.set_ylabel('BTTDA')
_=paired_plot(results, "XDAWN+RG", "BTTDA_rt_8", ax2, legend=False)
ax2.set_ylabel('')
ax2.set_xticks([])
ax2.set_yticks([])


_=paired_plot(results, "sLDA", "BTTDA_rt_8", ax3, legend=False)
ax3.set_ylabel('')
ax3.set_xticks([])
ax3.set_yticks([])

_=paired_plot(results, "tLDA", "BTTDA_rt_8", ax4, legend=False)
ax4.set_ylabel('')
ax4.set_xticks([])
ax4.set_yticks([])


fig.set_size_inches(7*0.7,4*0.7)
fig.tight_layout()
plt.savefig('roc_auc_results.pgf', bbox_inches='tight', pad_inches=0)